# PromptAlign G0: GitHub → Kaggle → VS Code
Bật Internet và GPU. Restart kernel rồi Run All. Gửi lại g0-*.zip và notebook có output. Đây là kiểm tra môi trường, chưa huấn luyện GenAI-Bench. Đổi DEVICE thành cpu nếu không có GPU.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/hoan25/BaiqiL-GenAI-Bench.git"
GIT_REF = "main"
DEVICE = "cuda" 
PRIVATE_REPO = False
ROOT = Path("/kaggle/working/promptalign-ai")

env = os.environ.copy()
env["GIT_TERMINAL_PROMPT"] = "0"
if PRIVATE_REPO:
    import base64
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    authorization = base64.b64encode(("x-access-token:" + token).encode()).decode()
    env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader", GIT_CONFIG_VALUE_0="AUTHORIZATION: basic " + authorization)
try:
    if not ROOT.exists():
        subprocess.run(["git", "clone", REPO_URL, str(ROOT)], env=env, check=True)
    origin = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=ROOT, text=True).strip()
    if origin != REPO_URL:
        raise RuntimeError("Existing directory belongs to a different repository")
    if subprocess.check_output(["git", "status", "--porcelain"], cwd=ROOT, text=True).strip():
        raise RuntimeError("Working tree has changes; preserve them before changing revision")
    subprocess.run(["git", "fetch", "origin", GIT_REF], cwd=ROOT, env=env, check=True)
    subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=ROOT, check=True)
finally:
    for key in ("GIT_CONFIG_COUNT", "GIT_CONFIG_KEY_0", "GIT_CONFIG_VALUE_0"):
        env.pop(key, None)
    if PRIVATE_REPO:
        del token, authorization
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip())


In [ ]:
result = subprocess.run([sys.executable, "-u", str(ROOT / "scripts/kaggle_g0.py"), "--device", DEVICE], cwd=ROOT)
from IPython.display import FileLink, display
for archive in sorted(Path("/kaggle/working").glob("g0-*.zip")):
    display(FileLink(str(archive)))
if result.returncode:
    raise RuntimeError("G0 did not pass. Send the ZIP logs above for diagnosis.")
